In [1]:
import ee
import geemap
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported.")

Libraries imported.


In [2]:
PROJECT_ID = "ee-chowdharypranav2005"

try:
    ee.Initialize(project=PROJECT_ID)
    print("✅ Google Earth Engine already authenticated.")
except Exception:
    print("Earth Engine authentication required...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("✅ Google Earth Engine authenticated.")

print(f"Project: {PROJECT_ID}")

✅ Google Earth Engine already authenticated.
Project: ee-chowdharypranav2005


In [3]:
DISTRICT_PATH = Path(
    "../data/raw/outbreaks/district_master.csv"
)

districts = pd.read_csv(
    DISTRICT_PATH
)

print("District master loaded.")
print("Shape:", districts.shape)

display(districts)

District master loaded.
Shape: (9, 6)


,State,District,Latitude,Longitude,Forest_Type,Elevation_Category
0,Karnataka,Shivamogga,13.9299,75.5681,Tropical Moist Deciduous,Medium
1,Karnataka,Uttara Kannada,14.7937,74.6869,Tropical Evergreen,Medium
2,Karnataka,Chikkamagaluru,13.3161,75.7720,Tropical Evergreen,High
3,Karnataka,Kodagu,12.4244,75.7382,Tropical Evergreen,High
4,Kerala,Wayanad,11.6854,76.1320,Tropical Evergreen,High
5,Kerala,Kannur,11.8745,75.3704,Tropical Moist Deciduous,Low
6,Goa,North Goa,15.4909,73.8278,Moist Deciduous,Low
7,Goa,South Goa,15.1170,74.1240,Moist Deciduous,Low
8,Maharashtra,Sindhudurg,16.3492,73.5594,Moist Deciduous,Medium


In [4]:
required_columns = [
    "State",
    "District",
    "Latitude",
    "Longitude"
]

missing_columns = [
    col
    for col in required_columns
    if col not in districts.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("✅ District master columns validated.")

✅ District master columns validated.


In [5]:
S2 = ee.ImageCollection(
    "COPERNICUS/S2_HARMONIZED"
)

CS_PLUS = ee.ImageCollection(
    "GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED"
)

QA_BAND = "cs_cdf"

CLEAR_THRESHOLD = 0.60

# radius
BUFFER_RADIUS_METERS = 50_000

# Regional reduction scale
REDUCTION_SCALE = 30

print("Satellite configuration ready.")
print("Cloud threshold:", CLEAR_THRESHOLD)
print("Buffer radius:", BUFFER_RADIUS_METERS / 1000, "km")
print("Reduction scale:", REDUCTION_SCALE, "m")

Satellite configuration ready.
Cloud threshold: 0.6
Buffer radius: 50.0 km
Reduction scale: 30 m


In [6]:
def build_annual_ndvi_collection(
    lat,
    lon,
    year,
    buffer_radius=BUFFER_RADIUS_METERS
):
    """
    Build a cloud-masked Sentinel-2 collection
    for one district-year.
    """

    lat = float(lat)
    lon = float(lon)
    year = int(year)

    point = ee.Geometry.Point(
        [lon, lat]
    )

    region = point.buffer(
        buffer_radius
    )

    start_date = ee.Date.fromYMD(
        year,
        1,
        1
    )

    end_date = ee.Date.fromYMD(
        year + 1,
        1,
        1
    )

    collection = (
        S2
        .filterBounds(region)
        .filterDate(
            start_date,
            end_date
        )
        .linkCollection(
            CS_PLUS,
            [QA_BAND]
        )
        .map(
            lambda image:
            image.updateMask(
                image
                .select(QA_BAND)
                .gte(CLEAR_THRESHOLD)
            )
        )
        .select(
            ["B4", "B8"]
        )
    )

    return collection, region

In [7]:
def extract_ndvi(
    lat,
    lon,
    year,
    buffer_radius=BUFFER_RADIUS_METERS,
    scale=REDUCTION_SCALE
):
    """
    Extract annual mean NDVI for one district-year.

    Returns
    -------
    dict
        NDVI
        Image_Count
    """

    collection, region = build_annual_ndvi_collection(
        lat=lat,
        lon=lon,
        year=year,
        buffer_radius=buffer_radius
    )

    image_count = int(
        collection
        .size()
        .getInfo()
    )

    if image_count == 0:

        return {
            "NDVI": np.nan,
            "Image_Count": 0
        }

    composite = (
        collection
        .median()
    )

    ndvi_image = (
        composite
        .normalizedDifference(
            ["B8", "B4"]
        )
        .rename("NDVI")
    )

    result = (
        ndvi_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=scale,
            bestEffort=True,
            maxPixels=1e9,
            tileScale=4
        )
        .get("NDVI")
        .getInfo()
    )

    if result is None:

        ndvi_value = np.nan

    else:

        ndvi_value = float(result)

    return {
        "NDVI": ndvi_value,
        "Image_Count": image_count
    }

In [8]:
ndvi_mean

NameError: name 'ndvi_mean' is not defined

In [9]:
test_locations = [
    {
        "Name": "Shivamogga 2018",
        "Latitude": 13.9299,
        "Longitude": 75.5681,
        "Year": 2018
    },

    {
        "Name": "Wayanad 2021",
        "Latitude": 11.6854,
        "Longitude": 76.1320,
        "Year": 2021
    },

    {
        "Name": "North Goa 2024",
        "Latitude": 15.4909,
        "Longitude": 73.8278,
        "Year": 2024
    }
]


test_results = []

for test in test_locations:

    print(
        f"Processing {test['Name']}..."
    )

    result = extract_ndvi(
        lat=test["Latitude"],
        lon=test["Longitude"],
        year=test["Year"]
    )

    test_results.append(
        {
            "Location": test["Name"],
            "NDVI": result["NDVI"],
            "Image_Count": result["Image_Count"]
        }
    )


test_df = pd.DataFrame(
    test_results
)

display(test_df)

Processing Shivamogga 2018...
Processing Wayanad 2021...
Processing North Goa 2024...


,Location,NDVI,Image_Count
0,Shivamogga 2018,0.394578,209
1,Wayanad 2021,0.515912,439
2,North Goa 2024,0.148292,301


In [10]:
print("Test observations:", len(test_df))

print(
    "Unique NDVI values:",
    test_df["NDVI"].nunique()
)

print(
    "Missing NDVI values:",
    test_df["NDVI"].isna().sum()
)

display(
    test_df.round(4)
)

Test observations: 3
Unique NDVI values: 3
Missing NDVI values: 0


,Location,NDVI,Image_Count
0,Shivamogga 2018,0.3946,209
1,Wayanad 2021,0.5159,439
2,North Goa 2024,0.1483,301


In [11]:
VISUAL_YEAR = 2021
VISUAL_DISTRICT = "Wayanad"

visual_row = districts[
    districts["District"] == VISUAL_DISTRICT
].iloc[0]

visual_lat = visual_row["Latitude"]
visual_lon = visual_row["Longitude"]

visual_collection, visual_region = (
    build_annual_ndvi_collection(
        lat=visual_lat,
        lon=visual_lon,
        year=VISUAL_YEAR
    )
)

visual_composite = (
    visual_collection
    .median()
)

visual_ndvi = (
    visual_composite
    .normalizedDifference(
        ["B8", "B4"]
    )
    .rename("NDVI")
)

Map = geemap.Map(
    center=[
        visual_lat,
        visual_lon
    ],
    zoom=8
)

ndvi_vis = {
    "min": 0.0,
    "max": 0.8,
    "palette": [
        "white",
        "yellow",
        "lightgreen",
        "green",
        "darkgreen"
    ]
}

Map.addLayer(
    visual_ndvi,
    ndvi_vis,
    f"{VISUAL_DISTRICT} NDVI {VISUAL_YEAR}"
)

Map.addLayer(
    visual_region,
    {},
    "50 km Study Region"
)

Map

Map(center=[np.float64(11.6854), np.float64(76.132)], controls=(WidgetControl(options=['position', 'transparen…

In [12]:
START_YEAR = 2015
END_YEAR = 2024

records = []

total_jobs = (
    len(districts)
    *
    (END_YEAR - START_YEAR + 1)
)

job_number = 0


for _, district in districts.iterrows():

    state = district["State"]
    district_name = district["District"]

    lat = float(
        district["Latitude"]
    )

    lon = float(
        district["Longitude"]
    )

    for year in range(
        START_YEAR,
        END_YEAR + 1
    ):

        job_number += 1

        print(
            f"[{job_number}/{total_jobs}] "
            f"{district_name} - {year}"
        )

        try:

            result = extract_ndvi(
                lat=lat,
                lon=lon,
                year=year
            )

            ndvi_value = result["NDVI"]
            image_count = result["Image_Count"]

        except Exception as error:

            print(
                f"⚠️ Error for "
                f"{district_name} {year}: "
                f"{error}"
            )

            ndvi_value = np.nan
            image_count = 0

        records.append(
            {
                "State": state,
                "District": district_name,
                "Year": year,
                "NDVI": ndvi_value,
                "Image_Count": image_count
            }
        )

[1/90] Shivamogga - 2015
[2/90] Shivamogga - 2016
[3/90] Shivamogga - 2017
[4/90] Shivamogga - 2018
[5/90] Shivamogga - 2019
[6/90] Shivamogga - 2020
[7/90] Shivamogga - 2021
[8/90] Shivamogga - 2022
[9/90] Shivamogga - 2023
[10/90] Shivamogga - 2024
[11/90] Uttara Kannada - 2015
[12/90] Uttara Kannada - 2016
[13/90] Uttara Kannada - 2017
[14/90] Uttara Kannada - 2018
[15/90] Uttara Kannada - 2019
[16/90] Uttara Kannada - 2020
[17/90] Uttara Kannada - 2021
[18/90] Uttara Kannada - 2022
[19/90] Uttara Kannada - 2023
[20/90] Uttara Kannada - 2024
[21/90] Chikkamagaluru - 2015
[22/90] Chikkamagaluru - 2016
[23/90] Chikkamagaluru - 2017
[24/90] Chikkamagaluru - 2018
[25/90] Chikkamagaluru - 2019
[26/90] Chikkamagaluru - 2020
[27/90] Chikkamagaluru - 2021
[28/90] Chikkamagaluru - 2022
[29/90] Chikkamagaluru - 2023
[30/90] Chikkamagaluru - 2024
[31/90] Kodagu - 2015
[32/90] Kodagu - 2016
[33/90] Kodagu - 2017
[34/90] Kodagu - 2018
[35/90] Kodagu - 2019
[36/90] Kodagu - 2020
[37/90] Kodagu - 

In [13]:
ndvi_audit_df = pd.DataFrame(
    records
)

print(
    "Shape:",
    ndvi_audit_df.shape
)

display(
    ndvi_audit_df.head(15)
)

Shape: (90, 5)


,State,District,Year,NDVI,Image_Count
0,Karnataka,Shivamogga,2015,0.501104,27
1,Karnataka,Shivamogga,2016,0.392206,187
2,Karnataka,Shivamogga,2017,0.428025,84
3,Karnataka,Shivamogga,2018,0.394578,209
4,Karnataka,Shivamogga,2019,0.391763,224
5,Karnataka,Shivamogga,2020,0.423659,218
6,Karnataka,Shivamogga,2021,0.432609,229
7,Karnataka,Shivamogga,2022,0.439875,223
8,Karnataka,Shivamogga,2023,0.445630,225
9,Karnataka,Shivamogga,2024,0.421058,227


In [14]:
print("=" * 70)
print("NDVI DATA QUALITY AUDIT")
print("=" * 70)

print(
    "\nRows:",
    len(ndvi_audit_df)
)

print(
    "\nMissing NDVI:",
    ndvi_audit_df["NDVI"]
    .isna()
    .sum()
)

print(
    "\nUnique NDVI values:",
    ndvi_audit_df["NDVI"]
    .nunique()
)

print(
    "\nNDVI descriptive statistics:"
)

display(
    ndvi_audit_df["NDVI"]
    .describe()
    .to_frame()
)

print(
    "\nImage-count statistics:"
)

display(
    ndvi_audit_df["Image_Count"]
    .describe()
    .to_frame()
)

NDVI DATA QUALITY AUDIT

Rows: 90

Missing NDVI: 0

Unique NDVI values: 90

NDVI descriptive statistics:


,NDVI
count,90.000000
mean,0.366698
std,0.153726
min,0.142723
25%,0.215025
50%,0.425842
75%,0.500059
max,0.625536



Image-count statistics:


,Image_Count
count,90.000000
mean,271.677778
std,125.013242
min,12.000000
25%,219.250000
50%,292.000000
75%,301.750000
max,530.000000


In [15]:
district_summary = (
    ndvi_audit_df
    .groupby("District")
    .agg(
        NDVI_Min=("NDVI", "min"),
        NDVI_Mean=("NDVI", "mean"),
        NDVI_Max=("NDVI", "max"),
        NDVI_Std=("NDVI", "std"),
        Mean_Image_Count=("Image_Count", "mean")
    )
    .round(4)
)

display(
    district_summary
)

,NDVI_Min,NDVI_Mean,NDVI_Max,NDVI_Std,Mean_Image_Count
District,,,,,
Chikkamagaluru,0.4321,0.4697,0.5711,0.0394,244.1
Kannur,0.1656,0.1729,0.1867,0.0075,255.1
Kodagu,0.5228,0.5523,0.6255,0.0292,248.6
North Goa,0.1427,0.1526,0.1661,0.0074,234.1
Shivamogga,0.3918,0.4271,0.5011,0.0326,185.3
Sindhudurg,0.2049,0.2279,0.2890,0.0236,235.7
South Goa,0.2631,0.2707,0.2893,0.0083,234.5
Uttara Kannada,0.4604,0.4883,0.5476,0.0249,422.3
Wayanad,0.5153,0.5388,0.6252,0.0337,385.4


In [16]:
year_summary = (
    ndvi_audit_df
    .groupby("Year")
    .agg(
        NDVI_Min=("NDVI", "min"),
        NDVI_Mean=("NDVI", "mean"),
        NDVI_Max=("NDVI", "max"),
        NDVI_Std=("NDVI", "std"),
        Mean_Image_Count=("Image_Count", "mean")
    )
    .round(4)
)

display(
    year_summary
)

,NDVI_Min,NDVI_Mean,NDVI_Max,NDVI_Std,Mean_Image_Count
Year,,,,,
2015,0.1661,0.4222,0.6255,0.1884,34.2222
2016,0.1505,0.3542,0.5228,0.1483,232.8889
2017,0.1519,0.3703,0.5593,0.1612,162.2222
2018,0.1468,0.3486,0.5284,0.1512,312.5556
2019,0.1474,0.3471,0.5317,0.1518,326.3333
2020,0.1427,0.3659,0.5601,0.1685,326.0000
2021,0.1516,0.3624,0.5462,0.1590,327.6667
2022,0.1620,0.3715,0.5578,0.1608,327.2222
2023,0.1588,0.3702,0.5551,0.1602,331.2222


In [17]:
ndvi_2015 = (
    ndvi_audit_df[
        ndvi_audit_df["Year"] == 2015
    ]
    [
        [
            "State",
            "District",
            "NDVI",
            "Image_Count"
        ]
    ]
)

display(
    ndvi_2015.round(4)
)

,State,District,NDVI,Image_Count
0,Karnataka,Shivamogga,0.5011,27
10,Karnataka,Uttara Kannada,0.5476,45
20,Karnataka,Chikkamagaluru,0.5711,36
30,Karnataka,Kodagu,0.6255,36
40,Kerala,Wayanad,0.6252,86
50,Kerala,Kannur,0.1852,42
60,Goa,North Goa,0.1661,12
70,Goa,South Goa,0.2893,12
80,Maharashtra,Sindhudurg,0.2890,12


In [18]:
duplicate_counts = (
    ndvi_audit_df["NDVI"]
    .round(8)
    .value_counts()
)

print(
    "Most frequently repeated NDVI values:"
)

display(
    duplicate_counts.head(10)
)

Most frequently repeated NDVI values:


NDVI
0.501104    1
0.392206    1
0.428025    1
0.394578    1
0.391763    1
0.423659    1
0.432609    1
0.439875    1
0.445630    1
0.421058    1
Name: count, dtype: int64

In [19]:
invalid_ndvi = ndvi_audit_df[
    (ndvi_audit_df["NDVI"] < -1)
    |
    (ndvi_audit_df["NDVI"] > 1)
]

print(
    "Values outside valid NDVI range:",
    len(invalid_ndvi)
)

if len(invalid_ndvi) > 0:
    display(invalid_ndvi)
else:
    print("All NDVI values are within [-1, 1].")

Values outside valid NDVI range: 0
All NDVI values are within [-1, 1].


In [20]:
no_imagery = ndvi_audit_df[
    ndvi_audit_df["Image_Count"] == 0
]

print(
    "District-years with zero usable images:",
    len(no_imagery)
)

if len(no_imagery) > 0:

    display(
        no_imagery
    )

else:

    print(
        "Every district-year has usable imagery."
    )

District-years with zero usable images: 0
Every district-year has usable imagery.


In [21]:
ndvi_dataset = (
    ndvi_audit_df[
        [
            "State",
            "District",
            "Year",
            "NDVI"
        ]
    ]
    .copy()
)

display(
    ndvi_dataset.head()
)

,State,District,Year,NDVI
0,Karnataka,Shivamogga,2015,0.501104
1,Karnataka,Shivamogga,2016,0.392206
2,Karnataka,Shivamogga,2017,0.428025
3,Karnataka,Shivamogga,2018,0.394578
4,Karnataka,Shivamogga,2019,0.391763


In [22]:
OUTPUT_PATH = Path(
    "../data/processed/"
    "ndvi_western_ghats_2015_2024.csv"
)

ndvi_dataset.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "Corrected NDVI dataset saved:"
)

print(
    OUTPUT_PATH
)

Corrected NDVI dataset saved:
..\data\processed\ndvi_western_ghats_2015_2024.csv


In [23]:
AUDIT_PATH = Path(
    "../data/processed/"
    "ndvi_western_ghats_2015_2024_audit.csv"
)

ndvi_audit_df.to_csv(
    AUDIT_PATH,
    index=False
)

print(
    "NDVI audit dataset saved:"
)

print(
    AUDIT_PATH
)

NDVI audit dataset saved:
..\data\processed\ndvi_western_ghats_2015_2024_audit.csv


In [24]:
print("=" * 72)
print("CORRECTED NDVI EXTRACTION COMPLETE")
print("=" * 72)

print(
    "\nDataset shape:",
    ndvi_dataset.shape
)

print(
    "Districts:",
    ndvi_dataset["District"].nunique()
)

print(
    "Years:",
    ndvi_dataset["Year"].min(),
    "-",
    ndvi_dataset["Year"].max()
)

print(
    "Unique NDVI values:",
    ndvi_dataset["NDVI"].nunique()
)

print(
    "Missing NDVI:",
    ndvi_dataset["NDVI"]
    .isna()
    .sum()
)

print(
    "\nNDVI range:"
)

print(
    "Minimum:",
    round(
        ndvi_dataset["NDVI"].min(),
        4
    )
)

print(
    "Maximum:",
    round(
        ndvi_dataset["NDVI"].max(),
        4
    )
)

print(
    "Mean:",
    round(
        ndvi_dataset["NDVI"].mean(),
        4
    )
)

print(
    "\nNotebook 08 complete."
)

CORRECTED NDVI EXTRACTION COMPLETE

Dataset shape: (90, 4)
Districts: 9
Years: 2015 - 2024
Unique NDVI values: 90
Missing NDVI: 0

NDVI range:
Minimum: 0.1427
Maximum: 0.6255
Mean: 0.3667

Notebook 08 complete.
